# Program 15: Retrieval chatbot for the Indian Penal Code

**Aim.** Download the Indian Penal Code, index it, and build a chatbot that answers questions with sources and keeps the conversation.

**Theory.** Retrieval-Augmented Generation: split the document into chunks, embed them, retrieve the chunks nearest to the question, and have the model answer using only those chunks. A rewrite step turns a follow-up like "and the punishment?" into a stand-alone question. Note: the IPC was replaced by the Bharatiya Nyaya Sanhita in July 2024, so this is a study exercise on the 1860 text, not legal advice.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import urllib.request
from pypdf import PdfReader
URL = 'https://www.mahapolice.gov.in/uploads/acts_rules/TheIndianPenalCode.pdf'   # copy published by the Maharashtra Police
PDF = DATA / 'ipc.pdf'
if not PDF.exists():
    urllib.request.urlretrieve(URL, PDF)
reader = PdfReader(str(PDF))
pages = [(i + 1, p.extract_text() or '') for i, p in enumerate(reader.pages)]
print(len(pages), 'pages,', sum(len(t.split()) for _, t in pages), 'words')

76 pages, 72567 words


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
split = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = []
for pg, text in pages:
    for c in split.split_text(text):
        chunks.append((pg, c))
print(len(chunks), 'chunks; first:', chunks[3][1][:160].replace('\n', ' '))

528 chunks; first: effect * * * throughout British India". The words and figures "on and from the first day of May, 1861" occurring  between the words "effect" and "throughout" we


In [4]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
M = emb.encode([c for _, c in chunks], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(M.shape[1]); index.add(np.asarray(M, dtype='float32'))
print(index.ntotal, 'vectors of', M.shape[1], 'dimensions')
def retrieve(q, k=4):
    s, i = index.search(emb.encode([q], normalize_embeddings=True).astype('float32'), k)
    return [(chunks[j][0], chunks[j][1], float(x)) for x, j in zip(s[0], i[0])]
for pg, c, s in retrieve('What is the punishment for theft?', 2):
    print(f'p.{pg} score {s:.2f}:', c[:200].replace('\n', ' '))

528 vectors of 384 dimensions
p.58 score 0.72: other, of having committed, or attempted to commit, an offence punishable with death or with 1*[imprisonment for  life], or with imprisonment for a term which may extend to ten years, shall be punishe
p.57 score 0.70: from Z to give. If A takes the property dishonestly, he commits theft. (p) A, in good faith, believing property  belonging to Z to be A's own property, takes that property out of B's possession. Here,


In [5]:
llm, label = get_llm()
print('model:', label)
history = []

def standalone(q):
    if not history: return q
    ctx = '\n'.join(f'Q: {a}\nA: {b[:200]}' for a, b in history[-3:])
    return ask(llm, f'Rewrite the last question so it makes sense alone. Output only the question.\n{ctx}\nLast question: {q}')

def chat(q):
    sq = standalone(q)
    hits = retrieve(sq, 4)
    context = '\n\n'.join(f'[page {p}] {c}' for p, c, _ in hits)
    ans = ask(llm, 'Answer using ONLY the context from the Indian Penal Code 1860. Cite section numbers if shown. '
                   'If the context does not contain the answer, say "The text I have does not say." This is not legal advice.\n\n'
                   f'CONTEXT:\n{context}\n\nQUESTION: {sq}')
    history.append((q, ans))
    return sq, ans, sorted({p for p, _, _ in hits})

for q in ['What is the punishment for theft?', 'And for robbery?', 'What is the capital of France?']:
    sq, ans, pgs = chat(q)
    print(f'USER: {q}\n(rewritten: {sq})\nBOT: {ans}\n(sources: pages {pgs})\n')

model: Groq openai/gpt-oss-20b (no Cohere key available)


USER: What is the punishment for theft?
(rewritten: What is the punishment for theft?)
BOT: **Punishment for theft (Section 379, Indian Penal Code, 1860)**  
Whoever commits theft shall be punished with imprisonment of either description for a term which may extend to **three years**, or with fine, or with both.
(sources: pages [57, 58])


USER: And for robbery?
(rewritten: What is the punishment for robbery?)
BOT: **Punishment for robbery (Section 392, Indian Penal Code, 1860)**  
- Whoever commits robbery shall be punished with **rigorous imprisonment for a term which may extend to ten years** and shall also be liable to a fine.  
- If the robbery is committed on the highway between sunset and sunrise, the imprisonment may be extended to **fourteen years**.
(sources: pages [58, 59])


USER: What is the capital of France?
(rewritten: What is the capital of France?)
BOT: The text I have does not say.
(sources: pages [5, 31, 44, 45])


## Observations

The follow-up "And for robbery?" works because of the rewrite step. The off-topic question should be refused, which tests whether the model sticks to the context. Retrieval can return the wrong section, so answers always show their pages.